In [ ]:
# ============================================================
# FULL TRAIN + DETERMINISTIC EVAL (EVERY N STEPS)
# MaskablePPO + Action Masking + Reward Norm
# ============================================================

import numpy as np
import gymnasium as gym

from stable_baselines3.common.callbacks import BaseCallback
from sb3_contrib.ppo_mask import MaskablePPO
from sb3_contrib.common.maskable.utils import get_action_masks

from envs.stack_gym import StackEnv

# ============================================================
# CONFIG
# ============================================================

config_dict = {
    "vessel_shape": (4, 4, 4),
    "yard_shape": (4, 4, 4),
    "num_containers": 64,
    "group_num": 4,
    "group_placement": "random",
    "observation_type": "stack_features",
    "reward_norm": True
}

config_dict_eval = {
     "vessel_shape": (4, 4, 4),
    "yard_shape": (4, 4, 4),
    "num_containers": 64,
    "group_num": 4,
    "group_placement": "random",
    "observation_type": "stack_features",
    "reward_norm": False
}

# ============================================================
# CUSTOM EVAL CALLBACK (DETERMINISTIC + MASKED)
# ============================================================

class MaskedEvalCallback(BaseCallback):
    def __init__(self, eval_env, eval_freq=25_000, n_eval_episodes=10, verbose=1):
        super().__init__(verbose)
        self.eval_env = eval_env
        self.eval_freq = eval_freq
        self.n_eval_episodes = n_eval_episodes

    def _on_step(self) -> bool:

        if self.n_calls % self.eval_freq == 0:

            rewards = []

            for _ in range(self.n_eval_episodes):

                obs, info = self.eval_env.reset()
                terminated = False
                truncated = False
                total_reward = 0.0

                while not (terminated or truncated):

                    action, _ = self.model.predict(
                        obs,
                        deterministic=True,
                        action_masks=get_action_masks(self.eval_env)
                    )

                    obs, reward, terminated, truncated, info = self.eval_env.step(action)
                    total_reward += reward

                rewards.append(total_reward)

            mean_reward = np.mean(rewards)
            var_reward = np.var(rewards)

            print("\n" + "=" * 60)
            print(f"Evaluation at step {self.num_timesteps} for {self.n_eval_episodes} episodes | Mean Reward: {mean_reward:.4f} | Variance Reward: {var_reward:.4f}")
            print("=" * 60)

        return True


# ============================================================
# CREATE TRAIN ENV
# ============================================================

train_env = StackEnv(config=config_dict, render_mode="rgb_array")

# ============================================================
# CREATE MODEL
# ============================================================

policy_kwargs = dict(net_arch=[256, 256, 32])

model = MaskablePPO(
    policy="MlpPolicy",
    env=train_env,
    learning_rate=3e-4,
    policy_kwargs=policy_kwargs,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    ent_coef=0.3,
    vf_coef=0.5,
    clip_range=0.2,
    verbose=1,
    device="cpu"
)

# ============================================================
# CREATE SEPARATE EVAL ENV
# ============================================================

eval_env = StackEnv(config=config_dict_eval, render_mode=None)

eval_callback = MaskedEvalCallback(
    eval_env=eval_env,
    eval_freq=25_000,      # evaluate every 25k steps
    n_eval_episodes=10
)

# ============================================================
# TRAIN
# ============================================================

print("\nTraining for 5,000,000 timesteps...\n")

model.learn(
    total_timesteps=5_000_000,
    callback=eval_callback
)

print("\nTraining complete.")

# ============================================================
# SAVE MODEL
# ============================================================

model.save("ppo_stack_masked_5M_with_reward_norm_bays4")
print("Model saved.")


In [ ]:
# ============================================================
# EVALUATE MODEL ACROSS 100 DIFFERENT SEEDS
# ============================================================

import numpy as np
from sb3_contrib.ppo_mask import MaskablePPO
from sb3_contrib.common.maskable.utils import get_action_masks

from envs.stack_gym import StackEnv

# ============================================================
# CONFIG (must match training config exactly)
# ============================================================

config_dict = {
     "vessel_shape": (4, 4, 4),
    "yard_shape": (4, 4, 4),
    "num_containers": 64,
    "group_num": 4,
    "group_placement": "random",
    "observation_type": "stack_features",
    "reward_norm": False
}

# ============================================================
# LOAD MODEL
# ============================================================

#model = MaskablePPO.load("ppo_stack_masked_5M_with_reward_norm_bays4")

print("=" * 70)
print("EVALUATING MODEL ACROSS 100 SEEDS")
print("=" * 70)

# ============================================================
# RUN 100 EPISODES WITH DIFFERENT SEEDS
# ============================================================

all_rewards = []
num_seeds = 100

for seed in range(num_seeds):
    
    # Create eval env
    eval_env = StackEnv(config=config_dict, render_mode=None)
    
    # Reset with specific seed
    obs, info = eval_env.reset(seed=seed)
    terminated = False
    truncated = False
    total_reward = 0.0
    step_count = 0
    
    # Run episode
    while not (terminated or truncated) and step_count < 120:
        action, _ = model.predict(
            obs,
            deterministic=True,
            action_masks=get_action_masks(eval_env)
        )
        
        obs, reward, terminated, truncated, info = eval_env.step(action)
        total_reward += reward
        step_count += 1
    
    all_rewards.append(total_reward)
    eval_env.close()
    
    # Progress update every 50 seeds
    if (seed + 1) % 50 == 0:
        print(f"Completed {seed + 1}/100 seeds...")

# ============================================================
# COMPUTE STATISTICS
# ============================================================

all_rewards = np.array(all_rewards)
mean_reward = np.mean(all_rewards)
std_reward = np.std(all_rewards)
min_reward = np.min(all_rewards)
max_reward = np.max(all_rewards)

print("\n" + "=" * 70)
print("EVALUATION RESULTS")
print("=" * 70)
print(f"Mean Reward: {mean_reward:.4f}")
print(f"Std Reward:  {std_reward:.4f}")
print(f"Min Reward:  {min_reward:.4f}")
print(f"Max Reward:  {max_reward:.4f}")
print("=" * 70)

In [ ]:
# ============================================================
# LOAD SAVED MODEL + VISUAL EVALUATION (MASKED + DETERMINISTIC)
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym

from sb3_contrib.ppo_mask import MaskablePPO
from sb3_contrib.common.maskable.utils import get_action_masks

from envs.stack_gym import StackEnv

# ============================================================
# CONFIG (must match training config exactly)
# ============================================================

config_dict = {
    "vessel_shape": (4, 4, 4),
    "yard_shape": (4, 4, 4),
    "num_containers": 64,
    "group_num": 4,
    "group_placement": "random",
    "seed": 200,
    "observation_type": "stack_features",
    "reward_norm": False
}

# ============================================================
# CREATE EVAL ENV
# ============================================================

eval_env = StackEnv(config=config_dict, render_mode="rgb_array")

# ============================================================
# LOAD MODEL
# ============================================================

#model = MaskablePPO.load("ppo_stack_masked_1.5M_with_reward_norm_bays3")
print("=" * 70)
print("TESTING TRAINED MODEL WITH VISUALIZATION")
print("=" * 70)

# ============================================================
# RUN ONE EPISODE
# ============================================================

obs, info = eval_env.reset(seed=200)
terminated = False
truncated = False
total_reward = 0.0
step_count = 0

while not (terminated or truncated) and step_count < 120:

    action, _ = model.predict(
        obs,
        deterministic=True,
        action_masks=get_action_masks(eval_env)
    )

    obs, reward, terminated, truncated, info = eval_env.step(action)

    total_reward += reward
    step_count += 1

    print(f"Step {step_count}: Action={action}, Reward={reward:.4f}, Total={total_reward:.4f}")

    # Render
    img = eval_env.render()

    if img is not None:
        plt.figure(figsize=(10, 7))
        plt.imshow(img)
        plt.title(
            f"Step {step_count} | Action: {action} | "
            f"Reward: {reward:.4f} | Total: {total_reward:.4f}"
        )
        plt.axis("off")
        plt.tight_layout()
        plt.show()
    else:
        print(f"Warning: No image rendered at step {step_count}")

# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("EPISODE FINISHED!")
print(f"Total Steps: {step_count}")
print(f"Total Reward: {total_reward:.4f}")
print("=" * 70)

eval_env.close()